# ClassWire TinyBERT NLU training

Before running: choose **GPU** under Notebook options and switch **Internet on**. Run every cell in order. The final cell creates one ZIP file containing the deployable INT8 ONNX artifact and measured reports.

In [ ]:
from pathlib import Path
import subprocess
repository = Path('/kaggle/working/ClassWire')
if repository.exists():
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/muneeb-anjum0/ClassWire.git', str(repository)], check=True)
%cd /kaggle/working/ClassWire
!python -m pip install -q -r ml/query_understanding/requirements.txt

In [ ]:
import platform
import torch
import transformers
print({"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training"

In [ ]:
!python -m ml.query_understanding.build_dataset --output ml/query_understanding/data/classwire_nlu.jsonl --total 24000 --seed 41
!python -m ml.query_understanding.validate_dataset ml/query_understanding/data/classwire_nlu.jsonl --report ml/query_understanding/reports/dataset_validation.json

In [ ]:
for seed in (41, 73, 109):
    subprocess.run([
        'python', '-m', 'ml.query_understanding.train',
        '--dataset', 'ml/query_understanding/data/classwire_nlu.jsonl',
        '--output', f'ml/query_understanding/checkpoints/candidate-{seed}',
        '--base-model', 'google/bert_uncased_L-8_H-256_A-4',
        '--epochs', '14', '--batch-size', '64', '--learning-rate', '4e-5',
        '--max-length', '96', '--patience', '4', '--intent-loss-weight', '2.0',
        '--slot-loss-weight', '1.2',
        '--slot-weight-power', '0.6', '--seed', str(seed), '--device', 'cuda',
    ], check=True)
!python -m ml.query_understanding.select_checkpoint --candidates ml/query_understanding/checkpoints/candidate-41 ml/query_understanding/checkpoints/candidate-73 ml/query_understanding/checkpoints/candidate-109 --output ml/query_understanding/checkpoints/best --report ml/query_understanding/reports/checkpoint_selection.json

In [ ]:
!python -m ml.query_understanding.evaluate --dataset ml/query_understanding/data/classwire_nlu.jsonl --checkpoint ml/query_understanding/checkpoints/best --split test --device cuda --output ml/query_understanding/reports/test_evaluation.json

In [ ]:
!python -m ml.query_understanding.export_onnx --checkpoint ml/query_understanding/checkpoints/best --output ml/query_understanding/output/v1 --model-version tinybert-nlu-v4

In [ ]:
!python -m ml.query_understanding.benchmark --dataset ml/query_understanding/data/classwire_nlu.jsonl --artifact ml/query_understanding/output/v1 --split test --runs 2000 --output ml/query_understanding/reports/cpu_benchmark.json

In [ ]:
from pathlib import Path
import shutil
delivery = Path('/kaggle/working/classwire_nlu_delivery')
delivery.mkdir(exist_ok=True)
shutil.copytree('ml/query_understanding/output/v1', delivery / 'artifact', dirs_exist_ok=True)
shutil.copytree('ml/query_understanding/reports', delivery / 'reports', dirs_exist_ok=True)
shutil.copy2('ml/query_understanding/data/classwire_nlu.manifest.json', delivery / 'reports/dataset_manifest.json')
shutil.copy2('ml/query_understanding/checkpoints/best/training_report.json', delivery / 'training_report.json')
archive = shutil.make_archive('/kaggle/working/classwire_nlu_delivery', 'zip', delivery)
print(f'Download: {archive}')